In [1]:
!apt-get update
!apt install -y python3.8

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2004/x86_64  InRelease [1581 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2004/x86_64  Packages [2790 kB]
Hit:3 http://archive.ubuntu.com/ubuntu focal InRelease
Get:4 http://packages.cloud.google.com/apt gcsfuse-focal InRelease [1227 B]
Get:5 http://security.ubuntu.com/ubuntu focal-security InRelease [128 kB]
Get:6 https://packages.cloud.google.com/apt cloud-sdk InRelease [1621 B]
Get:7 http://archive.ubuntu.com/ubuntu focal-updates InRelease [128 kB]
Err:4 http://packages.cloud.google.com/apt gcsfuse-focal InRelease
  The following signatures couldn't be verified because the public key is not available: NO_PUBKEY C0BA5CE6DC6315A3
Err:6 https://packages.cloud.google.com/apt cloud-sdk InRelease
  The following signatures couldn't be verified because the public key is not available: NO_PUBKEY C0BA5CE6DC6315A3
Get:8 https://packages.cloud.google.com/apt google-fast-socket InRelease [1071 B]
Err

In [2]:
!pip install virtualenv

%cd /kaggle/working
!virtualenv venv -p $(which python3.8)
# !virtualenv myenv

/kaggle/working
created virtual environment CPython3.8.10.final.0-64 in 1089ms
  creator CPython3Posix(dest=/kaggle/working/venv, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, setuptools=bundle, wheel=bundle, via=copy, app_data_dir=/root/.local/share/virtualenv)
    added seed packages: pip==22.3.1, setuptools==65.6.3, wheel==0.38.4
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator


In [3]:
!python3.8 --version

Python 3.8.10


In [4]:
# @title Enter the GitHub repository URL
github_url = "https://github.com/gavirttt/BertGCN" # @param {type:"string"}
import os
repo_name = github_url.split('/')[-1].replace('.git', '')
if not os.path.exists(repo_name):
  !git clone -b refactor/experiment-pipeline {github_url}
else:
  print(f"Repository '{repo_name}' already exists.")

Cloning into 'BertGCN'...
remote: Enumerating objects: 428, done.
remote: Counting objects: 100% (187/187), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 428 (delta 107), reused 126 (delta 57), pack-reused 241 (from 2)
Receiving objects: 100% (428/428), 54.44 MiB | 26.21 MiB/s, done.
Resolving deltas: 100% (233/233), done.


In [5]:
%cd BertGCN

/kaggle/working/BertGCN


In [6]:
!/kaggle/working/venv/bin/pip install torch==2.2.1+cu118 torchaudio==2.2.1+cu118 torchvision==0.17.1+cu118 torchdata==0.7.1 --index-url https://download.pytorch.org/whl/cu118
!/kaggle/working/venv/bin/pip install transformers datasets nltk scipy pytorch-ignite scikit-learn pydantic tqdm emoji>=2.8.0 wordcloud umap-learn==0.5.4 matplotlib
!/kaggle/working/venv/bin/pip install dgl -f https://data.dgl.ai/wheels/cu118/repo.html
!/kaggle/working/venv/bin/python -c "import torch, torchdata, dgl; print(f'Torch version:        {torch.__version__}'); print(f'Torch CUDA available: {torch.cuda.is_available()}'); print(f'Torch CUDA version:   {torch.version.cuda}'); print(f'TorchData version:    {torchdata.__version__}'); print(f'DGL version:          {dgl.__version__}')"

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.2/819.2 MB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 60.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 75.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 MB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 18.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 4.4 MB/s eta 0:00:00
  Obtaining dependency information for networkx from https://files.pythonhosted.org/packages/a8/05/9d4f9b78ead6b2661d6e8ea772e111fc4a9fbd866ad0c81906c11206b55e/networkx-3.1-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 11.2 MB/s eta 0:00:00
  Obtaining dependen

In [7]:
%%writefile inference.py
from prepare_twt_dataset import clean_text, load_author_lookup
from tqdm import tqdm
import os
import torch as th
import numpy as np
import pandas as pd
from model import BertClassifier

bert_init = 'dost-asti/RoBERTa-tl-sentiment-analysis'
nb_class = 3
batch_size = 128
label_map = {0: 'positive', 1: 'negative', 2: 'neutral'}
device = th.device('cuda' if th.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

checkpoint_path = '/kaggle/input/models/gabrielluigivirtucio/dost-astiroberta-tl-sentiment-analysis/transformers/finetuned/1/RoBERTa-tl-sentiment-analysis_twitter/checkpoint.pth'

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
print(f"Using checkpoint: {checkpoint_path}")

# Load and clean unlabeled data
df = pd.read_csv('data/tweets_unlabeled_set.csv')
load_author_lookup("data/well_known_authors_philippine_elections.csv")
df['cleaned_text'] = df['text'].apply(clean_text)
texts = df['cleaned_text'].tolist()
print(f"Loaded {len(texts)} unlabeled tweets")

# Load model and checkpoint
print("Loading model...")
model = BertClassifier(pretrained_model=bert_init, nb_class=nb_class)
ckpt = th.load(checkpoint_path, map_location=device)
model.bert_model.load_state_dict(ckpt['bert_model'])
model.classifier.load_state_dict(ckpt['classifier'])
model.eval()
model = model.to(device)

# Tokenize in batches
def tokenize_in_batches(texts, tokenizer, max_length=128, batch_size=128):
    all_input_ids = []
    all_attention_masks = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch,
            max_length=max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        all_input_ids.append(encoded['input_ids'])
        all_attention_masks.append(encoded['attention_mask'])
    return th.cat(all_input_ids, dim=0), th.cat(all_attention_masks, dim=0)

print("Tokenizing...")
input_ids, attention_mask = tokenize_in_batches(texts, model.tokenizer)
input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)

# Run inference
print("Running inference...")
all_probs = []
with th.no_grad():
    for i in tqdm(range(0, len(texts), batch_size), desc='Inference', unit='batch'):
        batch_input_ids = input_ids[i:i+batch_size]
        batch_attention_mask = attention_mask[i:i+batch_size]
        logits = model(batch_input_ids, batch_attention_mask)
        probs = th.nn.Softmax(dim=1)(logits).cpu().numpy()
        all_probs.append(probs)

all_probs = np.concatenate(all_probs, axis=0)
preds = all_probs.argmax(axis=1)

# Save results
df['sentiment'] = [label_map[p] for p in preds]
df['prob_positive'] = all_probs[:, 0]
df['prob_negative'] = all_probs[:, 1]
df['prob_neutral']  = all_probs[:, 2]

df.to_csv('predictions.csv', index=False)
print(f"Predictions saved to predictions.csv")
print(f"\nPrediction distribution:")
print(df['sentiment'].value_counts())

Overwriting inference.py


In [8]:
!/kaggle/working/venv/bin/python inference.py 

Using device: cuda
Using checkpoint: /kaggle/input/models/gabrielluigivirtucio/dost-astiroberta-tl-sentiment-analysis/transformers/finetuned/1/RoBERTa-tl-sentiment-analysis_twitter/checkpoint.pth
  [author lookup] 246 hashid → username mappings loaded from 'data/well_known_authors_philippine_elections.csv'.
Loaded 69335 unlabeled tweets
Loading model...
tokenizer_config.json: 1.36kB [00:00, 7.92MB/s]
vocab.json: 469kB [00:00, 23.5MB/s]
merges.txt: 269kB [00:00, 68.3MB/s]
tokenizer.json: 1.25MB [00:00, 117MB/s]
config.json: 100%|██████████████████████████████| 908/908 [00:00<00:00, 137kB/s]
pytorch_model.bin: 100%|██████████████████████| 436M/436M [00:02<00:00, 151MB/s]
Some weights of RobertaModel were not initialized from the model checkpoint at dost-asti/RoBERTa-tl-sentiment-analysis and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Toke

In [9]:
!/kaggle/working/venv/bin/python topic_modeling/data_prep.py \
  --labeled data/tweets_labeled_set.csv \
  --predictions predictions.csv \
  --authors data/well_known_authors_philippine_elections.csv \
  --output data/tweets_labeled_full.csv

[Stage 1] Loading labeled CSV: data/tweets_labeled_set.csv
  Rows    : 4464
  Columns : ['pseudo_id', 'text', 'retweetCount', 'replyCount', 'likeCount', 'quoteCount', 'viewCount', 'bookmarkCount', 'createdAt', 'lang', 'isReply', 'pseudo_conversationId', 'pseudo_inReplyToUsername', 'pseudo_author_userName', 'author_isBlueVerified', 'sentiment']
  Cleaning text for labeled rows...
  [author lookup] 246 hashid → username mappings loaded from 'data/well_known_authors_philippine_elections.csv'.

[Stage 1] Loading predictions CSV: predictions.csv
  Rows    : 69335
  Columns : ['pseudo_id', 'text', 'retweetCount', 'replyCount', 'likeCount', 'quoteCount', 'viewCount', 'bookmarkCount', 'createdAt', 'lang', 'isReply', 'pseudo_conversationId', 'pseudo_inReplyToUsername', 'pseudo_author_userName', 'author_isBlueVerified', 'cleaned_text', 'sentiment', 'prob_positive', 'prob_negative', 'prob_neutral']

  No overlapping pseudo_ids between datasets.

[Stage 1] Merging datasets...

MERGED DATASET SUMMA